# Class 5. Quantum Data Encoding and Feature Maps (Exercises)

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Normalize vectors for amplitude encoding.
2. Expand angle-encoded states analytically.
3. Compute and interpret a quantum kernel as an overlap.
4. Build small quantum Gram matrices in Qiskit.
5. Compare simple feature maps, including a re-uploading variant.
6. Cross-check one fidelity computation in PennyLane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

import pennylane as qml

## Conceptual notes for this session

Qubits start in $\lvert 0\rangle$ by default. Angle encoding uses classical features as gate parameters, for example $x_i \mapsto R_y(x_i)$ or $R_z(x_i)$, while amplitude encoding places data directly in amplitudes of
$$
\lvert\psi(x)\rangle=\sum_{j=0}^{2^n-1} a_j\lvert j\rangle,\qquad \sum_j\lvert a_j\rvert^2=1.
$$
A practical two-qubit amplitude-preparation recipe is to split probability mass by one rotation on the first qubit and then use controlled rotations on the second qubit to match branch-wise amplitude ratios.

For similarity design, Euclidean distance is magnitude-sensitive, cosine similarity is direction-sensitive, and quantum kernels are overlap-based:
$$
K_Q(x,y)=\left|\langle\phi(x)\vert\phi(y)\rangle\right|^2.
$$
There is no universal ranking that makes one always superior; usefulness depends on alignment with task geometry.

Angle encoding does not require vector normalization for physical validity because every real angle defines a unitary gate. However, scaling still matters due to periodicity: different feature values can map to the same state up to periodic aliasing. Phase encoding has no observable effect if applied only as $R_z(\theta)\lvert 0\rangle$ (global phase), but it becomes relevant after superposition and interference.

A two-qubit pure state has six real degrees of freedom after normalization and global-phase removal. The effective number of encoded features is therefore limited mainly by the chosen circuit ansatz, not by the Hilbert-space dimension alone.

---
## Part 1: Math tasks

### M5.1. Amplitude encoding

Given
$$
x = [0.4, 0.2, 0.8, 0.4],
$$
normalize the vector and write the corresponding two-qubit amplitude-encoded state.

In [ ]:
x_amp = np.array([0.4, 0.2, 0.8, 0.4], dtype=float)
norm = ...  # YOUR CODE HERE
psi_amp = ...  # YOUR CODE HERE

print(f"Norm = {norm:.6f}")
print("Normalized amplitudes:", np.round(psi_amp, 6))

### M5.2. Two-qubit angle encoding

Let
$$
x = \left(\frac{\pi}{6}, \frac{\pi}{2}\right).
$$
Compute the state
$$
\lvert\psi(x)\rangle = R_y(x_1)\lvert0\rangle \otimes R_y(x_2)\lvert0\rangle
$$
by expanding it in the computational basis.

In [ ]:
x1 = np.pi / 6
x2 = np.pi / 2

coeffs = np.array([
    ...,
    ...,
    ...,
    ...,
])  # YOUR CODE HERE

print("Coefficients [|00>, |01>, |10>, |11>]:")
print(np.round(coeffs, 6))
print("Norm:", np.linalg.norm(coeffs))

### M5.3. One-qubit quantum kernel

For the one-qubit encoding
$$
\lvert\phi(t)\rangle = R_y(t)\lvert 0\rangle,
$$
compute the quantum kernel for
$$
t = \frac{\pi}{5}, \qquad s = \frac{2\pi}{5}.
$$
Use both the closed-form formula and a direct overlap computation.

In [ ]:
t = np.pi / 5
s = 2 * np.pi / 5

ket_t = np.array([np.cos(t / 2), np.sin(t / 2)], dtype=complex)
ket_s = np.array([np.cos(s / 2), np.sin(s / 2)], dtype=complex)

kernel_formula = ...  # YOUR CODE HERE
kernel_overlap = ...  # YOUR CODE HERE

print(f"Kernel from formula: {kernel_formula:.6f}")
print(f"Kernel from overlap: {kernel_overlap:.6f}")

---
## Part 2: Programming tasks

### P5.1. Build encoded states in Qiskit

Create one basis-encoded state, one amplitude-encoded state, and one two-qubit angle-encoded state. Display their statevectors.

In [ ]:
basis_state = ...  # YOUR CODE HERE
amp_state = ...  # YOUR CODE HERE

qc_angle = QuantumCircuit(2)
qc_angle.ry(0.7, 0)
qc_angle.ry(1.1, 1)
angle_state = ...  # YOUR CODE HERE

print("Basis state |10>:")
print(np.round(basis_state.data, 6))
print()
print("Amplitude state:")
print(np.round(amp_state.data, 6))
print()
print("Angle-encoded state:")
print(np.round(angle_state.data, 6))

### P5.2. Build a small quantum Gram matrix with the adjoint trick

Use one-qubit angle encoding and estimate the symmetric Gram matrix for
`angles = [0.15, 0.9, 1.35, 2.1]`.

Use Qiskit and the probability of measuring `0` after `R_y(y)` followed by `R_y(-x)`.

The key identity is
$$
K_Q(x,y)=\left|\langle 0\vert U(x)^\dagger U(y)\vert 0\rangle\right|^2=P(0).
$$
For pure states this is also the fidelity,
$$
F\left(\lvert\phi(x)\rangle,\lvert\phi(y)\rangle\right)=\left|\langle\phi(x)\vert\phi(y)\rangle\right|^2.
$$
So this task computes each kernel entry as a Born-rule projection probability.

In [ ]:
sampler = StatevectorSampler()
angles = np.array([0.15, 0.9, 1.35, 2.1])


def adjoint_prob0_exact(x: float, y: float) -> float:
    qc = QuantumCircuit(1)
    qc.ry(y, 0)
    qc.ry(-x, 0)
    return ...  # YOUR CODE HERE

K = np.array([
    [... for y in angles]
    for x in angles
], dtype=float)  # YOUR CODE HERE

print("Quantum Gram matrix:\n", np.round(K, 6))

plt.figure(figsize=(4, 4))
plt.imshow(K, vmin=0.0, vmax=1.0, cmap="viridis")
plt.colorbar(shrink=0.8)
plt.title("One-qubit quantum Gram matrix")
plt.xlabel("j")
plt.ylabel("i")
plt.tight_layout()
plt.show()

### P5.3. Compare a single-pass feature map with re-uploading

Build two one-qubit encodings on the grid `values = np.linspace(0.0, 2.4, 6)`:

1. single pass: `R_y(x)`
2. re-uploading: `R_y(x) -> R_z(0.4x) -> R_y(theta1) -> R_y(x) -> R_z(0.4x) -> R_y(theta2)`

Compare the two kernel matrices.

Interpretation: the circuit is the feature map. Changing gate families or re-upload structure changes the induced overlap geometry, so the learner sees a different notion of similarity. This is the same design logic used in richer maps such as ZZ-style feature maps on multiple qubits.

In [ ]:
def state_single(x: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    return ...  # YOUR CODE HERE


def state_reupload(x: float, theta1: float = 0.5, theta2: float = -0.3) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)
    qc.rz(0.4 * x, 0)
    qc.ry(theta1, 0)
    qc.ry(x, 0)
    qc.rz(0.4 * x, 0)
    qc.ry(theta2, 0)
    return ...  # YOUR CODE HERE


def kernel_matrix(values: np.ndarray, state_fn) -> np.ndarray:
    states = [state_fn(v) for v in values]
    return np.array([
        [... for sb in states]
        for sa in states
    ], dtype=float)  # YOUR CODE HERE

values = np.linspace(0.0, 2.4, 6)
K_single = kernel_matrix(values, state_single)
K_reupload = kernel_matrix(values, state_reupload)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(
    axes,
    [K_single, K_reupload],
    ["Single-pass", "Re-uploading"],
):
    im = ax.imshow(mat, vmin=0.0, vmax=1.0, cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("j")
    ax.set_ylabel("i")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
plt.tight_layout()
plt.show()

### P5.4. Short PennyLane comparison

Use PennyLane to prepare `R_y(theta)|0>` and verify one pairwise fidelity against the Qiskit result.

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def qml_state(theta):
    qml.RY(theta, wires=0)
    return qml.state()

qa = np.array(qml_state(0.9), dtype=complex)
qb = np.array(qml_state(1.4), dtype=complex)
f_qml = ...  # YOUR CODE HERE
f_qiskit = ...  # YOUR CODE HERE

print(f"Qiskit fidelity:    {f_qiskit:.6f}")
print(f"PennyLane fidelity: {f_qml:.6f}")

### P5.5 (optional). Compare Euclidean, cosine, and quantum overlap on one dataset

Use the same small set of 2D inputs and compare three pairwise notions:

- Euclidean distance
- cosine similarity
- quantum overlap kernel induced by a product angle map
$$
\lvert\phi(x)\rangle = R_y(x_1)\lvert 0\rangle \otimes R_y(x_2)\lvert 0\rangle.
$$

Inspect where the orderings agree and where they differ.

In [ ]:
vectors = np.array([
    [0.2, 0.4],
    [0.9, 0.5],
    [1.4, 1.2],
], dtype=float)
labels = ["x0", "x1", "x2"]


def product_angle_state(v: np.ndarray) -> np.ndarray:
    qc = QuantumCircuit(2)
    qc.ry(float(v[0]), 0)
    qc.ry(float(v[1]), 1)
    return ...  # YOUR CODE HERE


states = [product_angle_state(v) for v in vectors]

print("Pairwise comparison on the same inputs")
for i in range(len(vectors)):
    for j in range(i + 1, len(vectors)):
        vi = vectors[i]
        vj = vectors[j]
        euclidean = ...  # YOUR CODE HERE
        cosine = ...  # YOUR CODE HERE
        quantum_overlap = ...  # YOUR CODE HERE
        print(
            f"{labels[i]} vs {labels[j]} | "
            f"d_E={euclidean:.6f} | "
            f"cos={cosine:.6f} | "
            f"K_Q={quantum_overlap:.6f}"
        )

---
## Summary

Encoding maps classical vectors to quantum states, and the kernel is the resulting overlap geometry. In classical methods one often chooses a kernel family directly, while in quantum methods one chooses an encoding circuit and obtains the kernel implicitly from state overlaps. Angle encoding does not require vector normalization for physical validity, but feature scaling still matters because periodic rotations can alias distant values. Entangling constructions, including ZZ-style maps, add interaction sensitivity beyond separable single-qubit encodings.